In [1]:
# --- Decision Tree using ONLY Gini Impurity (no "gain" printed) ---
# Dataset: Weekend / Weather / Parents / Money -> Decision
# Colab-ready. No networkx / diagrams. Includes interactive prediction.

from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Union

import pandas as pd
import numpy as np

# ---------------------------
# 1) Dataset
# ---------------------------
data = [
    {"Weekend":"W1",  "Weather":"Sunny", "Parents":"Yes", "Money":"Rich", "Decision":"Cinema"},
    {"Weekend":"W2",  "Weather":"Sunny", "Parents":"No",  "Money":"Rich", "Decision":"Tennis"},
    {"Weekend":"W3",  "Weather":"Windy", "Parents":"Yes", "Money":"Rich", "Decision":"Cinema"},
    {"Weekend":"W4",  "Weather":"Rainy", "Parents":"Yes", "Money":"Poor", "Decision":"Cinema"},
    {"Weekend":"W5",  "Weather":"Rainy", "Parents":"No",  "Money":"Rich", "Decision":"Stay In"},
    {"Weekend":"W6",  "Weather":"Rainy", "Parents":"Yes", "Money":"Poor", "Decision":"Cinema"},
    {"Weekend":"W7",  "Weather":"Windy", "Parents":"No",  "Money":"Poor", "Decision":"Cinema"},
    {"Weekend":"W8",  "Weather":"Windy", "Parents":"No",  "Money":"Rich", "Decision":"Shopping"},
    {"Weekend":"W9",  "Weather":"Windy", "Parents":"Yes", "Money":"Rich", "Decision":"Cinema"},
    {"Weekend":"W10", "Weather":"Sunny", "Parents":"No",  "Money":"Rich", "Decision":"Tennis"},
]
df = pd.DataFrame(data)
target = "Decision"
features = ["Weather","Parents","Money"]

print("Dataset")
display(df)

Dataset


,Weekend,Weather,Parents,Money,Decision
0,W1,Sunny,Yes,Rich,Cinema
1,W2,Sunny,No,Rich,Tennis
2,W3,Windy,Yes,Rich,Cinema
3,W4,Rainy,Yes,Poor,Cinema
4,W5,Rainy,No,Rich,Stay In
5,W6,Rainy,Yes,Poor,Cinema
6,W7,Windy,No,Poor,Cinema
7,W8,Windy,No,Rich,Shopping
8,W9,Windy,Yes,Rich,Cinema
9,W10,Sunny,No,Rich,Tennis


In [2]:
# ---------------------------
# 2) Gini impurity helpers (no 'gain' anywhere)
# ---------------------------
def gini_impurity_of_labels(labels: List[Any]) -> float:
    """Gini(S) = 1 - sum_i (p_i)^2"""
    n = len(labels)
    counts = Counter(labels)
    return 1.0 - sum((c/n)**2 for c in counts.values())

def weighted_child_gini(df: pd.DataFrame, feature: str, target: str) -> Tuple[float, Dict[Any, float]]:
    """
    For a multiway split by 'feature', compute:
      - weighted child Gini = sum_v (n_v/n)*Gini(S_v)
      - per-branch Gini dict
    """
    n = len(df)
    per_branch = {}
    weighted = 0.0
    for val, sub in df.groupby(feature):
        gi = gini_impurity_of_labels(sub[target].tolist())
        per_branch[val] = gi
        weighted += (len(sub)/n) * gi
    return weighted, per_branch

# Show root Gini and per-feature weighted child Gini (for transparency),
# but do NOT compute/print "gain".
root_gini = gini_impurity_of_labels(df[target].tolist())
rows = []
for feat in features:
    w_gini, parts = weighted_child_gini(df, feat, target)
    rows.append({
        "Feature": feat,
        "Gini(root)": round(root_gini, 3),
        "Weighted child Gini(feature)": round(w_gini, 3),
        "Per-branch Gini": {k: round(v, 3) for k, v in parts.items()}
    })
summary = pd.DataFrame(rows).sort_values("Weighted child Gini(feature)")
print("Gini summary (lower weighted child Gini is better):")
display(summary)

Gini summary (lower weighted child Gini is better):


,Feature,Gini(root),Weighted child Gini(feature),Per-branch Gini
1,Parents,0.58,0.360,"{'No': 0.72, 'Yes': 0.0}"
0,Weather,0.58,0.417,"{'Rainy': 0.444, 'Sunny': 0.444, 'Windy': 0.375}"
2,Money,0.58,0.486,"{'Poor': 0.0, 'Rich': 0.694}"


In [3]:
# ---------------------------
# 3) Tree structure (using ONLY Gini impurity to choose splits)
#     - choose feature with minimal weighted child Gini
# ---------------------------
@dataclass
class Leaf:
    prediction: Any
    counts: Dict[Any, int]

@dataclass
class DecisionNode:
    feature: str
    branches: Dict[Any, Any]  # value -> subtree

Tree = Union[Leaf, DecisionNode]

def majority_label(labels: List[Any]) -> Any:
    return Counter(labels).most_common(1)[0][0]

def build_tree_gini_only(
    df: pd.DataFrame,
    features: List[str],
    target: str,
    min_improvement: float = 1e-12
) -> Tree:
    labels = df[target].tolist()
    # Pure node → leaf
    if len(set(labels)) == 1:
        return Leaf(prediction=labels[0], counts=dict(Counter(labels)))
    # No features left → majority leaf
    if not features:
        return Leaf(prediction=majority_label(labels), counts=dict(Counter(labels)))

    # Current node impurity
    curr_gini = gini_impurity_of_labels(labels)

    # Choose the feature with the LOWEST weighted child Gini
    best_feat = None
    best_weighted = float("inf")
    for feat in features:
        w_gini, _ = weighted_child_gini(df, feat, target)
        if w_gini < best_weighted:
            best_weighted = w_gini
            best_feat = feat

    # Stop if no effective improvement
    if curr_gini - best_weighted <= min_improvement:
        return Leaf(prediction=majority_label(labels), counts=dict(Counter(labels)))

    # Recurse on branches
    branches = {}
    for val, sub in df.groupby(best_feat):
        if sub.empty:
            branches[val] = Leaf(prediction=majority_label(labels), counts=dict(Counter(labels)))
        else:
            remaining = [f for f in features if f != best_feat]
            branches[val] = build_tree_gini_only(sub, remaining, target, min_improvement=min_improvement)

    return DecisionNode(feature=best_feat, branches=branches)

tree = build_tree_gini_only(df, features, target)
print("Tree built using Gini impurity minimization (no 'gain' used).")

Tree built using Gini impurity minimization (no 'gain' used).


In [4]:
# ---------------------------
# 4) Pretty-print tree as text
# ---------------------------
def pretty_print_tree(tree: Tree, indent: str = ""):
    if isinstance(tree, Leaf):
        print(indent + f"Leaf: predict={tree.prediction}  counts={tree.counts}")
    else:
        print(indent + f"[Split on: {tree.feature}]")
        for val, subtree in tree.branches.items():
            print(indent + f" ├─ {val}")
            pretty_print_tree(subtree, indent + " │   ")

print("\nHuman-readable tree:")
pretty_print_tree(tree)


Human-readable tree:
[Split on: Parents]
 ├─ No
 │   [Split on: Weather]
 │    ├─ Rainy
 │    │   Leaf: predict=Stay In  counts={'Stay In': 1}
 │    ├─ Sunny
 │    │   Leaf: predict=Tennis  counts={'Tennis': 2}
 │    ├─ Windy
 │    │   [Split on: Money]
 │    │    ├─ Poor
 │    │    │   Leaf: predict=Cinema  counts={'Cinema': 1}
 │    │    ├─ Rich
 │    │    │   Leaf: predict=Shopping  counts={'Shopping': 1}
 ├─ Yes
 │   Leaf: predict=Cinema  counts={'Cinema': 5}


In [5]:
# ---------------------------
# 5) Predict function + evaluation
# ---------------------------
def predict_one(tree: Tree, row: dict) -> Any:
    if isinstance(tree, Leaf):
        return tree.prediction
    feat = tree.feature
    val = row.get(feat)
    if val in tree.branches:
        return predict_one(tree.branches[val], row)
    # unseen value → majority fallback
    leaves = []
    def collect_leaves(t):
        if isinstance(t, Leaf):
            leaves.append(t)
        else:
            for sb in t.branches.values():
                collect_leaves(sb)
    collect_leaves(tree)
    agg = Counter()
    for lf in leaves:
        agg.update(lf.counts)
    return agg.most_common(1)[0][0]

y_true = df[target].tolist()
y_pred = [predict_one(tree, row) for row in df.to_dict(orient="records")]
acc = np.mean([t==p for t,p in zip(y_true, y_pred)])
print(f"\nTraining accuracy: {acc*100:.1f}%")


Training accuracy: 100.0%


In [6]:
# ---------------------------
# 6) Simple user input → prediction
# ---------------------------
# Normalize helper (capitalize first letter, or Yes/No proper casing)
def normalize_choice(value: str) -> str:
    v = value.strip().lower()
    # Map to known sets
    if v in {"sunny","rainy","windy"}:
        return v.capitalize()
    if v in {"yes","no"}:
        return v.capitalize()
    if v in {"rich","poor"}:
        return v.capitalize()
    return value  # leave as-is if unexpected

print("\n--- Interactive Prediction ---")
print("Valid choices:")
print("  Weather ∈ {Sunny, Rainy, Windy}")
print("  Parents ∈ {Yes, No}")
print("  Money   ∈ {Rich, Poor}")

try:
    user_weather = normalize_choice(input("Enter Weather (Sunny/Rainy/Windy): "))
    user_parents = normalize_choice(input("Enter Parents (Yes/No): "))
    user_money   = normalize_choice(input("Enter Money (Rich/Poor): "))

    user_row = {"Weather": user_weather, "Parents": user_parents, "Money": user_money}
    pred = predict_one(tree, user_row)
    print(f"\nPrediction for {user_row} ⇒ Decision = {pred}")
except EOFError:
    # If running in a non-interactive environment, show a demo instead
    demo = {"Weather": "Windy", "Parents": "No", "Money": "Rich"}
    print("\nInput not available; demo row used:", demo)
    print("Decision =", predict_one(tree, demo))


--- Interactive Prediction ---
Valid choices:
  Weather ∈ {Sunny, Rainy, Windy}
  Parents ∈ {Yes, No}
  Money   ∈ {Rich, Poor}
Enter Weather (Sunny/Rainy/Windy): Windy
Enter Parents (Yes/No): Yes
Enter Money (Rich/Poor): Poor

Prediction for {'Weather': 'Windy', 'Parents': 'Yes', 'Money': 'Poor'} ⇒ Decision = Cinema
